# Study Assistant using RAG Llama3.1:8b (LangChain Version)

In this notebook, we build a RAG pipeline that function as a study assistant. The assistant is fed with personal notes and a textbook, and answers questions grounded in that material to make finding information easier. The project is built with Llama 3.1:8b as the generator, delivered **with LangChain** to demonstrate how its framework abstractions can implement the same pipeline with less code.

Here is the pipeline overview to easily navigate the project:

**Ingestion**

PDF Files → Load Documents (`PyPDFLoader`) → Chunk Text (`RecursiveCharacterTextSplitter`) → Embed Chunks (`HuggingFaceEmbeddings`) → Store in ChromaDB

**Query**

User Query → Retrieve Top-k Chunks → Build Prompt → Generate Answer (`OllamaLLM`) → Return Answer + Sources


Similarly, we begin with setting up the environment by importing necessary libraries.

In [1]:
# Run this cell once to install dependencies if needed
!pip install langchain langchain-community langchain-chroma langchain-huggingface langchain-ollama pypdf --break-system-packages -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scispacy 0.5.5 requires spacy<3.8.0,>=3.7.0, but you have spacy 3.8.7 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
# Import necessary libraries

import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import re
import pandas as pd

## 1. Data ingestion

The PDFs are extracted with `PyPDFLoader`, a pre-built loader that opens a PDF and returns a list of `Document` objects, one per page, each containing the text content and metadata. This replaces the manual extraction loop used in the raw-Python version. LangChain's loader produces plain, per-page text extraction, so cleaning still needs to be applied afterward to fix hyphens and remove extra line breaks and produce clean output.

In [7]:
# Add file paths
PDF_PATHS = [
    "./data_source/personal_notes.pdf",
    "./data_source/textbook.pdf",
]

def extract_pdf_pages(path):
    """Extracts text from a list of PDF file paths and returns a list of Document objects."""

    loader = PyPDFLoader(path)
    raw_docs = loader.load() # loader extracts one Document per page
    for d in raw_docs:
        # applied the same cleaning
        cleaned_text = re.sub(r'(\w)-\n(\w)', r'\1\2', d.page_content) # fix hyphen breaks
        cleaned_text = re.sub(r'\n', ' ', cleaned_text).strip()        # remove newlines and extra spaces
        d.page_content = cleaned_text

    return raw_docs

documents = []
for path in PDF_PATHS:
    documents.extend(extract_pdf_pages(path))
print(f"Loaded {len(documents)} pages from {len(PDF_PATHS)} file(s).")

# Preview
print("\n\nPreview of the extracted page")
print("="*50)
print(documents[0].page_content[:500])
print("\nMetadata:", documents[0].metadata)

Loaded 73 pages from 2 file(s).


Preview of the extracted page
Text representation A. Bag-of-words (BOW) and One-hot encoding Bag-of-Words (BoW):A text representation technique where a document is represented as a  vector of word counts or occurrences in a vocabulary. • One-Hot Encoding:A technique that converts categorical variables (including words) into binary  vectors, where only one position in the vector is "hot" (1), and the rest are "cold" (0). • Example D1 -“I am very happy today” D2 -“I am not well and not happy today” D3 -“I wish I could go to pl

Metadata: {'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext, AppendMode 1.1', 'creator': 'OneNote', 'creationdate': "D:20260824103240Z00'00'", 'moddate': "D:20260824103329Z00'00'", 'source': './data_source/personal_notes.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}


## 2. Text chunking

We continue by splitting the text using `RecursiveCharacterTextSplitter`. It splits text by trying a list of separators in order (paragraph breaks, then sentences, then words), falling back to a hard character-count split only if nothing else fits. This improves on the raw fixed-size word count splitting approach, as this splitter respects natural text boundaries first, rather than always cutting at a fixed word count. It also automatically carries over each chunk's source metadata that makes it more efficient to code than the raw-Python version.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,      # characters, not words — roughly 350-400 words
    chunk_overlap=200,    # overlap between chunks to maintain context
    separators=["\n\n", "\n", ". ", " ", ""],  # tries paragraph, then line, then sentence, then word
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} document(s).")

# Preview
print("\n\nExample chunk")
print("="*50)
print(chunks[0].page_content[:400])
print("\nMetadata:", chunks[0].metadata)


Created 223 chunks from 73 document(s).


Example chunk
Text representation A. Bag-of-words (BOW) and One-hot encoding Bag-of-Words (BoW):A text representation technique where a document is represented as a  vector of word counts or occurrences in a vocabulary. • One-Hot Encoding:A technique that converts categorical variables (including words) into binary  vectors, where only one position in the vector is "hot" (1), and the rest are "cold" (0). • Exam

Metadata: {'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext, AppendMode 1.1', 'creator': 'OneNote', 'creationdate': "D:20260824103240Z00'00'", 'moddate': "D:20260824103329Z00'00'", 'source': './data_source/personal_notes.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}


## 3. Text encoding and vector database

In here, we transform our text into semantic vectors using **`all-MiniLM-L6-v2`** and store them in the **ChromaDB** vector database. As we can see, using LangChain we can define the same embedding model, and it handles **embedding and storing in a single function call**, without needing to write a manual loop or manage metadata separately.

In [10]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_langchain",  # separate folder from the raw-Python version
    collection_name="study_assistant_lc",
)

print(f"Indexed {vectorstore._collection.count()} chunks into ChromaDB (via LangChain).")

Indexed 223 chunks into ChromaDB (via LangChain).


## 4. Retriever

The retriever returns the top-k most similar chunks for a given query. With LangChain, we turn the vector store directly into a retriever object using `.as_retriever()`, and call `.invoke()` to pass in a query. This single call handles embedding the query, searching the vector store, and returning the top results together with their metadata, rather than the multiple manual steps used in the raw-Python version.

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Add a sample query
test_query = "What is lemmatization?"
hits = retriever.invoke(test_query)

for i, doc in enumerate(hits, 1):
    page = doc.metadata.get("page", "?")
    source = doc.metadata.get("source", "unknown")
    print(f"[{i}] {source} (page {page})")
    print(doc.page_content[:250])
    print()


[1] ./data_source/textbook.pdf (page 19)
. How is lemmatization done? The most sophisticated methods for lemmatization involve complete morphological parsing of the word. Morphology is the study of the way words are built up from smaller meaning-bearing units called morphemes.morpheme Two b

[2] ./data_source/personal_notes.pdf (page 3)
. Lemmatization & Stemming The goal of both stemming and lemmatization is to reduce inflectional forms and sometimes  derivationally related forms of a word to a common base form. Lemmatizationis the task of determining that two words have the same r

[3] ./data_source/personal_notes.pdf (page 4)
. Lemmatization & Stemming The goal of both stemming and lemmatization is to reduce inflectional forms and sometimes  derivationally related forms of a word to a common base form. Lemmatizationis the task of determining that two words have the same r



## 5. Building prompt

We build the prompt using `PromptTemplate`, which serves the same purpose as before: instructing the LLM and combining the retrieved context with the question. Using LangChain, we standardize the prompt as a reusable template object, so it can be composed directly with the rest of the pipeline (retriever, LLM, output parser) as a single chain in the next step.

In [13]:
prompt_template = PromptTemplate.from_template("""You are a study assistant. Answer the question using ONLY the context below.
If the context does not contain enough information to answer, say so clearly instead of guessing.

When citing information, use ONLY the bracket number, like [1] or [2], directly after the relevant
sentence. Do NOT repeat the source filename or page numbers in your answer text. A source list
will be shown separately after your answer.

Write your answer as clear, direct prose. Do not describe what each numbered source says one by
one and synthesize the information into a single coherent answer.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(
        f"[{i+1}] (Source: {d.metadata.get('source', 'unknown')}, page {d.metadata.get('page', '?')})\n{d.page_content}"
        for i, d in enumerate(docs)
    )

print(prompt_template.format(context="(context goes here)", question=test_query))


You are a study assistant. Answer the question using ONLY the context below.
If the context does not contain enough information to answer, say so clearly instead of guessing.

When citing information, use ONLY the bracket number, like [1] or [2], directly after the relevant
sentence. Do NOT repeat the source filename or page numbers in your answer text. A source list
will be shown separately after your answer.

Write your answer as clear, direct prose. Do not describe what each numbered source says one by
one and synthesize the information into a single coherent answer.

Context:
(context goes here)

Question: What is lemmatization?

Answer:


## 6. Full pipeline

For generation, we previously had to define an ask() function to encapsulate three separate steps called in sequence:
```
retrieval → prompt → generation
```

LCEL (LangChain Expression Language) allows us to compose the full pipeline into a single |-chained expression, following this sequence:
```
retriever -> format_docs -> prompt_template -> llm -> parse output
```
Notice that this chain already includes a parse output using `StrOutputParser`(), to extract the plain text answer from the LLM's response object to keeps the final output as a clean string.

In [ ]:
# Define LLM model
llm = OllamaLLM(model="llama3.1:8b")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke(test_query)
print(answer)


Lemmatization is the task of determining that two words have the same root, despite their surface differences [2]. It involves reducing inflectional forms and sometimes derivationally related forms of a word to a common base form [3].


## 7. Full pipeline with citations

Since LCEL chains don't return sources by default (the chain above only outputs the final answer string), we retrieve documents separately, alongside invoking the chain, so that both the answer and its sources can be returned together.

In [ ]:
def format_page(doc):
    return f"page {doc.metadata.get('page', '?')}"

def ask(query, k=3, verbose=True):
    retrieved_docs = retriever.invoke(query)
    answer = rag_chain.invoke(query)

    if verbose:
        print("QUESTION:", query)
        print("\nANSWER:\n" + answer)
        print("\nSOURCES:")
        for i, d in enumerate(retrieved_docs, 1):
            print(f"  [{i}] {d.metadata.get('source', 'unknown')}, {format_page(d)}")

    return {"query": query, "answer": answer, "sources": retrieved_docs}

_ = ask("What is lemmatization?")


QUESTION: What is lemmatization?

ANSWER:
Lemmatisization is the task of determining that two words have the same root, despite their surface differences [2]. It involves reducing inflectional forms and sometimes derivationally related forms of a word to a common base form [3].

SOURCES:
  [1] textbook.pdf, page 19
  [2] personal_notes.pdf, page 1
  [3] personal_notes.pdf, page 8


## 8. Evaluation set

Similarly, we evaluate the LangChain RAG pipeline using the same set of queries, to compare its performance against the raw-Python version.
Below, we begin with a set of **questions whose answers we have confirmed are present in the source texts.** 

In [ ]:
# Define evaluation queries that have been confirmed to have relevant information in the indexed documents
eval_set = [
    {
        "question": "What is one-hot vector?",
        "expected_source_contains": "vector",
    },
    {
        "question": "What is sentence segmentation and how to apply it?",
        "expected_source_contains": "breaking",
    },
    {
        "question": "What are the tasks performed in text normalization?",
        "expected_source_contains": "tokenizing",
    },
    {
        "question": "How does feedforward work in word prediction?",
        "expected_source_contains": "probability",
    },
]

eval_results = []
for item in eval_set:
    result = ask(item["question"], verbose=False)
    retrieved_text = " ".join(d.page_content.lower() for d in result["sources"])
    retrieval_hit = item["expected_source_contains"].lower() in retrieved_text

    sources_str = ";\n".join(
        f"[{i}] {d.metadata.get('source', 'unknown')}, {format_page(d)}"
        for i, d in enumerate(result["sources"], 1)
    )

    eval_results.append({
        "Question": item["question"],
        "Retrieval hit": True if retrieval_hit else False,
        "Answer": result["answer"],
        "Sources": sources_str,
    })

df = pd.DataFrame(eval_results)
pd.set_option("display.max_colwidth", None)

styled = df.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
display(styled)

accuracy = sum(1 for r in eval_results if r["Retrieval hit"] == True) / len(eval_results)
print(f"\nRetrieval hit rate: {accuracy:.0%}")

,Question,Retrieval hit,Answer,Sources
0,What is one-hot vector?,True,"A one-hot vector is a vector that has one element equal to 1, in the dimension corresponding to that word's index in the vocabulary, while all the other elements are set to zero [1].","[1] textbook.pdf, page 40; [2] personal_notes.pdf, page 10; [3] personal_notes.pdf, page 15"
1,What is sentence segmentation and how to apply it?,True,"Sentence segmentation is the task of breaking up a text into individual sentences, using cues like periods or exclamation points [3]. It is one of the tasks included in text normalization [1].","[1] textbook.pdf, page 12; [2] textbook.pdf, page 17; [3] textbook.pdf, page 1"
2,What are the tasks performed in text normalization?,True,"According to the context, at least three tasks are commonly applied as part of any normalization process: 1. Tokenizing (segmenting) words [1]. 2. Normalizing word formats. 3. Segmenting sentences.","[1] textbook.pdf, page 12; [2] textbook.pdf, page 25; [3] textbook.pdf, page 25"
3,How does feedforward work in word prediction?,True,"A feedforward neural language model takes as input a representation of some number of previous words and outputs a probability distribution over possible next words [1]. It approximates the probability of a word given the entire prior context by approximating based on the N-1 previous words [1]. For example, in a 4-gram example, the neural net estimates the probability P(wt = i|wt−3,wt−2,wt−1) [1]. Neural language models use embeddings to represent words in the prior context, allowing them to generalize better to unseen data [1]. During training, the model predicts each word in a text, using the cross-entropy loss to update its weights [2]. The model assigns a probability to the correct next word, and the cross-entropy loss is calculated as the negative log of this probability [2].","[1] textbook.pdf, page 39; [2] textbook.pdf, page 49; [3] textbook.pdf, page 26"



Retrieval hit rate: 100%


In [ ]:
# Add questions that has no answer from source text
unanswerable_eval_set = [
    "What is overfitting?",
    "What year was the transformer architecture published?",
    "How does quantum computing relate to neural networks?",
    "What is the capital of France?",
]

unanswerable_results = []
for question in unanswerable_eval_set:
    result = ask(question, verbose=False)
    answer_lower = result["answer"].lower()

    unanswerable_results.append({
        "Question": question,
        "Answer": result["answer"],
    })

df_unanswerable = pd.DataFrame(unanswerable_results)
pd.set_option("display.max_colwidth", None)
styled_unanswerable = df_unanswerable.style.set_properties(
    **{"white-space": "pre-wrap", "text-align": "left"}
)
display(styled_unanswerable)

,Question,Answer
0,What is overfitting?,There is no mention of overfitting in the provided context.
1,What year was the transformer architecture published?,There is no information in the provided context about the transformer architecture or its publication year.
2,How does quantum computing relate to neural networks?,There is no information in the provided context about quantum computing and its relationship to neural networks.
3,What is the capital of France?,There is no information in the provided context that answers the question about the capital of France.


We can see that the pipeline performs correctly on both answerable and unanswerable queries that retrieves the right chunks when a fact exists in the source texts, and appropriately declining when it doesn't. 

This confirms that **the LangChain implementation works as intended**, replicating the raw-Python pipeline's with requiring noticeably less code. The two implementations use the same overall approach and often produce comparable results, but they are not identical, as differences in the processing steps resulted in variations in chunking and cleaning outcomes between the two.

## 9. Adding conversational memory

As we have already confirmed that the initial prompt works accordingly, we want to extend the application into multi-turn conversation. The `ask()` function above is only capable of answering questions independently, and therefore it has no awareness of what was asked before and is unable to handle follow-up questions. We want to extend the application by enabling users to continue with follow-ups to enhance the study assistant.

To support real conversation, we add two things on top of the existing pipeline:

1. **Query condensation**: Before retrieval, we make an extra LLM call that rewrites the follow-up into a standalone question, using the recent conversation as context (e.g. *"can you give a simple example of that?"* → *"Can you give a simple example of a one-hot vector?"*).  This enables retrieval to work well, since the condensed version is what actually gets used to search for the answer in the vector store.
2. **History in the final prompt**: The final prompt used to call the LLM also includes the recent conversation history, so the LLM can reference earlier context when composing its reply, not just the retrieved chunks.

Therefore, the multi-turn version calls the LLM **twice**, one extra call compared to single-turn support, but this makes the study assistant considerably more useful.

In [ ]:
def format_history(messages):
    """Formats recent chat turns as plain text (e.g. 'User: ...\nAssistant: ...') for the LLM to read as prior context. Keeps only the last few turns."""

    if not messages:
        return "(no previous conversation)"
    
    return "\n".join(f"{role.capitalize()}: {msg}" for role, msg in messages)

def condense_question(query, history_messages):
    """Rewrites a follow-up question (e.g. 'what about the second one?') into a standalone question, so retrieval can actually search for it. 
    This to get the top-k relevant chunks based on the follow-up question, not the previous conversation."""

    if not history_messages:
        return query

    history_text = format_history(history_messages)
    condense_prompt = f"""Given this conversation history:
{history_text}

And this follow-up question: "{query}"

Rewrite the follow-up as a standalone question that makes sense without the conversation history.
If it's already standalone, return it unchanged. Reply with ONLY the rewritten question, nothing else."""

    return llm.invoke(condense_prompt).strip()

# Extend the prompt template with a `history` block
prompt_template_with_memory = PromptTemplate.from_template("""You are a study assistant. Answer the question using ONLY the context below.
If the context does not contain enough information to answer, say so clearly instead of guessing.

When citing information, use ONLY the bracket number, like [1] or [2], directly after the relevant
sentence. Do NOT repeat the source filename or page numbers in your answer text. A source list
will be shown separately after your answer.

Write your answer as clear, direct prose. Do not describe what each numbered source says one by
one and synthesize the information into a single coherent answer.

Conversation so far (for context only. answer the current question, not previous ones):
{history}

Context:
{context}

Question: {question}

Answer:""")

chain_with_memory = prompt_template_with_memory | llm | StrOutputParser()

def ask_with_memory(query, history_messages, k=3, verbose=True):
    
    standalone_query = condense_question(query, history_messages)
    retrieved_docs = retriever.invoke(standalone_query)
    context = format_docs(retrieved_docs)
    history_text = format_history(history_messages)

    answer = chain_with_memory.invoke({"history": history_text,"context": context,"question": query,})

    if verbose:
        print("QUESTION:", query)
        if standalone_query != query:
            print("(condensed to:", standalone_query, ")")
        print("\nANSWER:\n" + answer)
        print("\nSOURCES:")
        for i, d in enumerate(retrieved_docs, 1):
            print(f"  [{i}] {d.metadata.get('source', 'unknown')}, {format_page(d)}")
            
    return {"query": query, "standalone_query": standalone_query, "answer": answer, "sources": retrieved_docs}

### Demo: A Multi-Turn Conversation

Below, we test the pipeline's ability to handle a follow-up question. We will see how the follow-up question gets condensed and whether the answer provided by the model still aligns well with the original follow-up question.

In [ ]:
history = []

# Question 1: a standalone question that works fine without any prior context
result1 = ask_with_memory("What is one-hot vector?", history)
history.append({"role": "user", "content": result1["query"]})
history.append({"role": "assistant", "content": result1["answer"]})

print("\n" + "="*50 + "\n")

# Question 2: a follow-up that only makes sense given previous context
result2 = ask_with_memory("Can you give a simple example of that?", history)
history.append({"role": "user", "content": result2["query"]})
history.append({"role": "assistant", "content": result2["answer"]})

QUESTION: What is one-hot vector?

ANSWER:
A one-hot vector is a vector that has one element equal to 1—in the dimension corresponding to that word’s index in the vocabulary— while all the other elements are set to zero. [1]

SOURCES:
  [1] textbook.pdf, page 40
  [2] personal_notes.pdf, page 12
  [3] personal_notes.pdf, page 14


QUESTION: Can you give a simple example of that?
(condensed to: Can you give a simple example of a one-hot vector? )

ANSWER:
A simple example of one-hot encoding is given as: "I am very happy today" where "happy" is encoded as a vector [0, 0, 0, 0, 1, 0, 0, 0] where the position corresponding to "happy" in the vocabulary is 1 and the rest are 0 [1].

SOURCES:
  [1] personal_notes.pdf, page 1
  [2] personal_notes.pdf, page 3
  [3] personal_notes.pdf, page 7


Based on the result, we can see that the condensed question rewrites the follow-up question by adding the missing context, so that the retrieval process is able to understand the query and retrieve the relevant chunks. Without this, it would most likely miss the relevant information entirely. At the same time, we can see both the answer for the standalone and follow-up question received correct answers.

**This multi-turn version is deployed to [Streamlit](http://desystudyassistant.streamlit.app) for application demonstration.**